#### Environment Setup and Model Initialization

In [2]:
import cv2
from ultralytics import YOLO

#### Initialize the Pre-trained YOLOv8 Model

In [ ]:
# This will automatically download 'yolov8n.pt'
model = YOLO('yolov8n.pt')

### Set up Video Input

In [ ]:
video_source = 0 
cap = cv2.VideoCapture(video_source)

#### Check if webcam opened successfully

In [ ]:
if not cap.isOpened():
    print("Error: Could not open video source.")
    exit()
    
# Press 'q' will exit the video stream
print("Press 'q' on your keyboard to exit the video stream.")

Press 'q' on your keyboard to exit the video stream.


#### Process the Video Frame-by-Frame

In [ ]:
while cap.isOpened():
    success, frame = cap.read()
    
    if not success:
        print("Video stream ended or failed to read frame.")
        break

    # Apply Detection AND Tracking simultaneously
    
    # Here persist=True tells the tracker to remember IDs from previous frames
    results = model.track(frame, persist=True, verbose=False)

    # Extract Tracking Information and Draw Layouts
    # Check if any objects were actually detected in the current frame
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int) # Bounding box coordinates
        ids = results[0].boxes.id.cpu().numpy().astype(int)     # Unique Tracking IDs
        clss = results[0].boxes.cls.cpu().numpy().astype(int)   # Class indices 
        
        # Loop through every detected object in this single frame
        for box, id_num, cls in zip(boxes, ids, clss):
            x1, y1, x2, y2 = box
            class_name = model.names[cls] # Convert index (e.g., 0) to string label (e.g., "person")
            
            # Draw the bounding box rectangle around the object (Green color)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            
            # Create a label string with the Object Class and its tracking ID
            label = f"{class_name} ID: {id_num}"
            
            # Draw background text label
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Display the Output Stream in Real-Time
    cv2.imshow("Real-Time Object Detection & Tracking", frame)

    # Stop the program smoothly if the user presses the 'q' key
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
# Clean up window and release hardware resources
cap.release()
cv2.destroyAllWindows()

WARNING not enough matching points
